<span style="color:blue">

# Data Treatment Train

<span>

In [1]:
# On importe les bibliothèques dont on aura besoin
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Titanic dataset load
data_train = pd.read_csv('Titanic/train.csv')

In [3]:
data_train.tail(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
881,882,0,3,"Markun, Mr. Johann",male,33.0,0,0,349257,7.8958,NaN,S
882,883,0,3,"Dahlberg, Miss. Gerda Ulrika",female,22.0,0,0,7552,10.5167,NaN,S
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,NaN,S
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.0500,NaN,S
885,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,NaN,Q
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.7500,NaN,Q


<span style="color:blue">

## Visualization of the dataset and pre-processing

<span>

<span style="color:blue">

<br>

At first glance, the `Cabin` column appears to contain many `NaN` values. Let's identify the columns containing these values and their proportion.

<br>

<span>

In [4]:
data_train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

<span style="color:blue">

<br>

The `Cabin` column contains numerous values, as do the `Age` and `Embarked` columns. Given the total data of 891 raw values, we chose to remove the `Cabin` column and the `NaN` values from the `Age` and `Embarked` columns. For `Age` and `Embarked` columns we will replace `NaN` values by the median value since we cannot suppress raws.

<br>

<span>

In [5]:
data_train["Embarked"] = data_train["Embarked"].fillna("S") # Random choice
data_train["Age"] = data_train["Age"].fillna(data_train["Age"].median())
data_train = data_train.drop(columns=["Cabin"])
data_train = data_train.reset_index(drop=True)

NameError: name 'data_test' is not defined

In [ ]:
data_train.head()

<span style="color:blue">

<br>

We can then transform the `Sex`  and `Embarked` columns using 0 and 1 instead of male and female. We can also try segmenting the `Age` and `Fare` columns into categories.

<br>

<span>

In [ ]:
# Visualizing the Age distribution
plt.figure(figsize=(8,5))
plt.hist(data_train["Age"], bins=50, color="lightblue", edgecolor="black")
plt.xlabel("Age")
plt.ylabel("Count")
plt.title("Age distribution")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8,5))
plt.hist(data_train["Fare"], bins=20, color="lightpink", edgecolor="black")
plt.xlabel("Fare")
plt.ylabel("Count")
plt.title("Fare distribution")
plt.grid(alpha=0.3)
plt.show()

<span style="color:blue">

<br>

Let's create 10-year categories for the "Age" column. And for the price, separate the categories \[0, 25], \[25, 100], \[100, 200], \[200, 300] and more.

<br>

<span>

In [ ]:
data_train["Embarked"].unique()

In [ ]:
# 0 and 1 for Sex and Embarked columns
data_train["Sex"] = data_train["Sex"].map({"male": 0, "female": 1})
data_train["Embarked"] = data_train["Embarked"].map({"S": 0, "C": 1, "Q": 2})

# Age category
thresholds_age = [0, 10, 20, 30, 40, 50, 60, 70, 80]
labels_age = [0, 1, 2, 3, 4, 5, 6, 7]
data_train["AgeCategory"] = pd.cut(data_train["Age"], bins=thresholds_age, labels=labels_age, include_lowest=True)
data_train = data_train.drop(columns=["Age"])

# Fare category
thresholds_fare = [0, 25, 100, 200, 300, 600]
labels_fare = [0, 1, 2, 3, 4]
data_train["FareCategory"] = pd.cut(data_train["Fare"], bins=thresholds_fare, labels=labels_fare, include_lowest=True)
data_train = data_train.drop(columns=["Fare"])

In [ ]:
data_train.tail()

<span style="color:blue">

<br>

The column `Ticket` does not appear to be very relevant for predicting whether a passenger survived or not.

<br>

<span>

In [ ]:
# A lot of unique tickets
data_train["Ticket"].unique()
data_train = data_train.drop(columns=["Ticket"])

<span style="color:blue">

<br>

## New Features

<br>

<span>

<span style="color:blue">

<br>

We can consider introducing new features. Since the `Name` doesn't identify survivors, we can instead obtain their title ("Mrs., Miss…"). This is often an indicator of social status. We can also assume that people without families would favor those who have families. Therefore, we can add a feature indicating whether a person is alone and the size of their family on board.

<br>

<span>

In [ ]:
# Extract Title from Name
data_train["Title"] = data_train["Name"].str.extract(' ([A-Za-z]+)\.', expand=False)

In [ ]:
data_train["Title"].value_counts()

In [ ]:
# Some title are very rare, let's gather them
rare_titles = ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", 
               "Sir", "Jonkheer", "Mlle", "Mme", "Ms"]

data_train["Title"] = data_train["Title"].replace(rare_titles, "Rare")

# Replacing str by values
data_train["Title"] = data_train["Title"].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})
data_train = data_train.drop(columns=["Name"])

In [ ]:
data_train.head()

In [ ]:
# Family is siblings/spouse + parents/children + self
data_train["FamilySize"] = data_train["SibSp"] + data_train["Parch"] + 1

data_train["IsAlone"] = (data_train["FamilySize"] == 1).astype(int)

data_train = data_train.drop(columns=["SibSp", "Parch"])

In [ ]:
data_train.head()

<span style="color:blue">

## Saving in X_train and y_train

<span>

In [ ]:
data_train.to_csv("Titanic/Titanic_train_processed.csv", index=False)